# Agent-Reach reel — GPU extras (generated stills + AI video inserts)

Independent of the voice/lip-sync notebook (run in either order, or in a second Kaggle session).
1. **RealVisXL V5 Lightning** (CreativeML OpenRAIL++-M, commercial use OK) makes 7 studio-style object photos on plain white (for cut-outs: eye, switchboard, cables, magnifier, newspapers, radar dish, rotary phone).
2. **CogVideoX-2b** (Apache-2.0) makes 3 short video inserts (eye opening, server-room dolly, hands on a switchboard), then ffmpeg interpolates 8 → 24 fps.
Everything generated is **AI imagery** and will be tagged "AI IMAGE" / "AI VIDEO" in the reel.

Kaggle: Accelerator **GPU P100** (or T4 x2), Internet **On**, Run All. About 40–60 min. Resumable. Output: `agentreach-extras.zip`.

In [ ]:
import os, sys, json, subprocess, shutil, glob, time
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.makedirs(f'{BASE}/stills', exist_ok=True); os.makedirs(f'{BASE}/video', exist_ok=True); os.chdir(BASE)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode and check: print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError(cmd)
    return r
sh('which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)')
sh('pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg safetensors')
STILLS = [
 ("eye",     "macro photograph of a single human eye, hazel iris, long lashes, soft studio light, plain light grey background", 11),
 ("switch",  "vintage telephone switchboard panel with brass jacks and cloth patch cords, studio product photo, seamless white background", 12),
 ("cables",  "tangle of colourful ethernet cables with plugs, studio product photo, seamless white background", 13),
 ("lens",    "old brass magnifying glass, studio product photo, seamless white background", 14),
 ("news",    "stack of folded newspapers tied with string, studio product photo, seamless white background", 15),
 ("radar",   "retro radar dish antenna on a short pole, studio product photo, seamless white background", 16),
 ("phone",   "vintage red rotary telephone, studio product photo, seamless white background", 17),
]
VIDEOS = [
 ("eye_open",   "extreme close-up of a human eye slowly opening, glowing blue data lines reflected in the iris, cinematic, shallow depth of field", 21),
 ("server_dolly","slow dolly forward through a dark server room, rows of softly blinking lights and thick cables, cinematic", 22),
 ("switch_hands","close-up of hands plugging cloth patch cords into a vintage telephone switchboard, warm light, cinematic", 23),
]
print('ready')

In [ ]:
# 1. Stills with RealVisXL V5.0 Lightning (fp16 on GPU, 6 steps) -> 1024x1024 PNGs
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler
from huggingface_hub import hf_hub_download
try:
    ckpt = hf_hub_download('SG161222/RealVisXL_V5.0_Lightning', 'RealVisXL_V5.0_Lightning_fp16.safetensors')
    pipe = StableDiffusionXLPipeline.from_single_file(ckpt, torch_dtype=torch.float16, use_safetensors=True).to('cuda')
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing='trailing')
    pipe.set_progress_bar_config(disable=True)
    NEG = 'cartoon, illustration, 3d render, painting, low quality, blurry, text, watermark, logo, deformed, oversaturated, cluttered background'
    for sid, prompt, seed in STILLS:
        out = f'{BASE}/stills/{sid}.png'
        if os.path.exists(out): continue
        t = time.time()
        for k in range(2):   # 2 candidates per subject; you / I pick later
            g = torch.Generator('cuda').manual_seed(seed * 10 + k)
            img = pipe(prompt=prompt + ', photorealistic, sharp focus, soft even studio lighting, high detail', negative_prompt=NEG, width=1024, height=1024,
                       num_inference_steps=6, guidance_scale=1.6, generator=g).images[0]
            img.save(f'{BASE}/stills/{sid}_{k}.png')
        shutil.copy(f'{BASE}/stills/{sid}_0.png', out)
        print(sid, f'{time.time()-t:.0f}s')
    del pipe; torch.cuda.empty_cache()
except Exception as e:
    print('STILLS FAILED:', repr(e)[:800])

In [ ]:
# 2. AI video inserts with CogVideoX-2b (fp16, 49 frames @ 8 fps, 720x480), then 24 fps by motion interpolation
import torch
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video
try:
    pipe = CogVideoXPipeline.from_pretrained('THUDM/CogVideoX-2b', torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload(); pipe.vae.enable_tiling(); pipe.vae.enable_slicing()
    for vid, prompt, seed in VIDEOS:
        raw = f'{BASE}/video/{vid}_raw.mp4'; out = f'{BASE}/video/{vid}.mp4'
        if os.path.exists(out): continue
        t = time.time()
        frames = pipe(prompt=prompt, num_videos_per_prompt=1, num_inference_steps=40, num_frames=49, guidance_scale=6,
                      generator=torch.Generator('cuda').manual_seed(seed)).frames[0]
        export_to_video(frames, raw, fps=8)
        sh(f"ffmpeg -v error -y -i {raw} -vf \"minterpolate=fps=24:mi_mode=mci:mc_mode=aobmc:vsbmc=1\" -c:v libx264 -crf 16 -pix_fmt yuv420p {out}")
        print(vid, f'{time.time()-t:.0f}s')
    del pipe; torch.cuda.empty_cache()
except Exception as e:
    print('VIDEO FAILED:', repr(e)[:800])

In [ ]:
# 3. Zip
sh(f'cd {BASE} && rm -f agentreach-extras.zip && zip -qr agentreach-extras.zip stills video -x "video/*_raw.mp4"')
print(sh(f'ls -l {BASE}/agentreach-extras.zip && ls stills video').stdout)

**Done.** Download `agentreach-extras.zip` from the Output panel and put it in the Drive folder. If a stage printed `FAILED`, paste that line back to me; the other stage's files are still in the zip.